# Experiment: Ablation Study — Optuna Parameter Optimization

**Ziel:** Optimale RAG-Parameter für System 1 (Monolith RAG) finden.  
**Methode:** Optuna TPE (Bayesian Optimization) mit 40–60 Trials.  
**Metriken:** RAGAS Context Precision, Recall, Faithfulness.  
**Output:** `configs/best_config.yaml`

### Suchraum
| Parameter | Wertebereich |
|---|---|
| `chunk_size` | [1000, 1500, 2000] |
| `overlap_pct` | [10%, 20%] |
| `bm25_weight` | [0.3, 0.5, 0.7] |
| `pre_rerank_top_k` | [15, 20, 25] |
| `post_rerank_top_k` | [3..8] |

---

## 1. Setup

In [ ]:
import logging
import os
from pathlib import Path

# Ensure CWD is the project root (needed for src imports)
PROJECT_ROOT = Path(__file__).resolve().parent.parent.parent if '__file__' in dir() else Path.cwd()
# Walk up until we find pyproject.toml
while not (PROJECT_ROOT / 'pyproject.toml').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

from src.common import setup_logging
from src.common.ingestion import ProcessedFiling, load_processed_filing
from src.evaluation.gold_standard_loader import load_gold_standard
from src.systems.rag_monolith.ablation.optuna_search import run_ablation
from src.systems.rag_monolith.ablation.results_analyzer import save_best_config, print_top_trials, generate_summary_table

setup_logging(logging.INFO)

# Paths (all relative to PROJECT_ROOT)
GOLD_STANDARD_PATH = PROJECT_ROOT / "notebooks" / "experiments" / "ablation_test_data.csv"
BEST_CONFIG_PATH = PROJECT_ROOT / "configs" / "best_config.yaml"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print(f"Gold standard: {GOLD_STANDARD_PATH.exists()}")
print(f"Processed dir: {PROCESSED_DIR.exists()}")

## 2. Daten laden

In [ ]:
# Load ALL processed filings from disk
TICKERS = ["AAPL", "MSFT", "AMZN", "GOOGL"]

filings: list[ProcessedFiling] = []
for ticker in TICKERS:
    ticker_dir = PROCESSED_DIR / ticker
    if not ticker_dir.exists():
        print(f"  ✗ {ticker}: Verzeichnis nicht gefunden")
        continue
    for md_file in sorted(ticker_dir.glob("10K_*.md")):
        filing_date = md_file.stem.replace("10K_", "")
        try:
            f = load_processed_filing(ticker, filing_date=filing_date)
            filings.append(f)
            print(f"  ✓ {ticker} ({filing_date}): {len(f.sections)} sections, {len(f.full_text):,} chars")
        except Exception as e:
            print(f"  ✗ {ticker} ({filing_date}): {e}")

print(f"\nLoaded {len(filings)} filings")

In [ ]:
# Load gold standard evaluation dataset
gold_standard = load_gold_standard(GOLD_STANDARD_PATH)
print(f"Gold standard: {len(gold_standard)} Q&A pairs")
print(f"Types: {set(item.query_type for item in gold_standard)}")

# Preview
for item in gold_standard[:3]:
    print(f"  [{item.query_type}] {item.question[:60]}... → {item.ground_truth}")

## 3a. Smoke Test

Validiert die gesamte Pipeline (Build → Query → RAGAS Eval) mit einer einzigen Frage.  
Läuft in <60s und fängt Konfigurationsfehler ab, bevor 50 Trials gestartet werden.

In [ ]:
from src.systems.rag_monolith.pipeline import MonolithRAGPipeline
from src.evaluation.ragas_evaluator import evaluate_run

# Build pipeline with default config
pipeline = MonolithRAGPipeline()
pipeline.build(filings)

# Run a single query
test_item = gold_standard[0]
result = pipeline.query(test_item.question)
print(f"Question: {test_item.question}")
print(f"Answer:   {result.answer[:200]}...")
print(f"Contexts: {len(result.contexts)} chunks retrieved")

# Run RAGAS evaluation on that single query
scores = evaluate_run(
    gold_standard=[test_item],
    answers=[result.answer],
    contexts=[result.contexts],
)
print(f"\n✅ Smoke test passed: {scores.to_dict()}")

## 3b. Optuna Ablation Study

Bayesian Optimization mit TPE-Sampler.  
SQLite-Storage für Resume-Fähigkeit bei Abbruch.  
**Nur starten wenn Smoke Test (3a) bestanden.**

In [ ]:
# Run ablation study (resume-capable via SQLite)
N_TRIALS = 50  # Empfohlen: 40-60

study = run_ablation(
    filings=filings,
    gold_standard=gold_standard,
    n_trials=N_TRIALS,
)

## 4. Ergebnisse

In [ ]:
import optuna
from pathlib import Path

# Generiere den absoluten Pfad zur Datenbank
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'pyproject.toml').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

db_path = PROJECT_ROOT / "data" / "ablation_study.db"
storage_url = f"sqlite:///{db_path.absolute()}"

print(f"Lade Datenbank von: {storage_url}")

# Lade die Study
study = optuna.load_study(
    study_name="rag_monolith_ablation", 
    storage=storage_url
)

print(f"✅ Study loaded! Totale fertige Trials: {len(study.trials)}")


In [ ]:
# Import fehlende Funktion
from src.systems.rag_monolith.ablation.results_analyzer import generate_summary_table

# Ergebnisse als DataFrame
import pandas as pd

rows = generate_summary_table(study)
df = pd.DataFrame(rows)
df.head(10)


## 5. Visualisierung

In [ ]:
## 5. Visualisierung (Robust mit Matplotlib)
import optuna
from optuna.visualization.matplotlib import plot_optimization_history
from optuna.visualization.matplotlib import plot_param_importances
from optuna.visualization.matplotlib import plot_parallel_coordinate
import matplotlib.pyplot as plt

# Wichtig für sauberes Layout in Notebooks
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

# Optimization History (Konvergiert der Score?)
fig1 = plot_optimization_history(study)
if fig1.figure._suptitle: fig1.figure.suptitle("") # Verhindert Overlap mit evtl. Figure-Titeln
fig1.set_title("Wird System 1 über die Trials besser?", pad=15)
fig1.figure.tight_layout()
plt.show()

# Parameter Importance (Was ist am wichtigsten?)
fig2 = plot_param_importances(study)
if fig2.figure._suptitle: fig2.figure.suptitle("")
fig2.set_title("Welcher Parameter beeinflusst den Score am stärksten?", pad=15)
fig2.figure.tight_layout()
plt.show()

# Parameter Relationships
fig3 = plot_parallel_coordinate(study)
if fig3.figure._suptitle: fig3.figure.suptitle("")
fig3.set_title("Welche Kombinationen funktionieren gut?", pad=15)
fig3.figure.set_size_inches(12, 6) 
fig3.figure.tight_layout()
plt.show()

## 6. Best Config exportieren

In [ ]:
# Export best configuration
config_path = save_best_config(study, output_path=BEST_CONFIG_PATH)

print(f"\n✅ Best config saved to: {config_path}")
print(f"\nBest parameters:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")
print(f"\nBest composite score: {study.best_value:.4f}")